# Activity 10: Duplicate and Key Quality Clinic

## Assignment

You are checking a small claims extract before it joins to a policy lookup table. Your job is to find duplicate rows, decide which business-key duplicates are safe to collapse, clean the policy lookup key, and prove the final join did not change the claim row count.

Documentation: [`duplicated`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html), [`drop_duplicates`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html), [`merge`](https://pandas.pydata.org/docs/reference/api/pandas.merge.html).

Solutions notebook for instructor reference. Keep the student drill notebook unchanged.

## Instructor Teaching Note

Do not frame duplicates as automatically bad. Ask students what the duplicate means: an exact repeated row, a later update to the same business event, or a lookup table key that must be unique.

In [ ]:
import pandas as pd

claims_path = "../Activity_10_Duplicate_Key_Quality/Resources/claims_key_quality.csv"
policy_path = "../Activity_10_Duplicate_Key_Quality/Resources/policy_lookup.csv"

## Example 1: Find duplicate rows

`duplicated(keep=False)` marks every row involved in a duplicate group. This is useful for auditing because it shows both copies, not only the later copy.

In [ ]:
example = pd.DataFrame({
    "claim_id": ["A1", "A2", "A2", "A3"],
    "amount": [100, 250, 250, 300],
})

example[example.duplicated(keep=False)]

## Example 2: Drop duplicates with a policy

`drop_duplicates` is not magic cleanup. You choose a `subset` and a `keep` rule. Here we keep the latest row after sorting by `updated_at`.

In [ ]:
updates = pd.DataFrame({
    "claim_id": ["A1", "A1", "A2"],
    "status": ["open", "paid", "review"],
    "updated_at": pd.to_datetime(["2026-01-01 09:00", "2026-01-02 10:00", "2026-01-01 11:00"]),
})

updates.sort_values("updated_at").drop_duplicates(subset=["claim_id"], keep="last")

## Your Turn 1: Load and inspect the claim extract

Read `claims_key_quality.csv`, parse `claim_date` and `updated_at`, then display the first rows and row count.

In [ ]:
claims_df = pd.read_csv(claims_path, parse_dates=["claim_date", "updated_at"])

claims_df.head()
print(len(claims_df))

## Your Turn 2: Find exact duplicate rows

Find rows that are exact duplicates across all columns. Then create `claims_no_exact_dupes` by dropping exact duplicates.

In [ ]:
exact_duplicate_rows = claims_df[claims_df.duplicated(keep=False)]
claims_no_exact_dupes = claims_df.drop_duplicates()

print(len(claims_df))
print(len(claims_no_exact_dupes))
exact_duplicate_rows

## Your Turn 3: Check business-key duplicates

Now check duplicates on the business key `policy_id`, `claim_date`, `claim_type`, and `claim_amount`. Sort by `updated_at`, then keep the last record for each business key.

In [ ]:
business_key = ["policy_id", "claim_date", "claim_type", "claim_amount"]

business_key_duplicate_rows = claims_no_exact_dupes[
    claims_no_exact_dupes.duplicated(subset=business_key, keep=False)
]

claims_sorted = claims_no_exact_dupes.sort_values("updated_at")
clean_claims_df = claims_sorted.drop_duplicates(subset=business_key, keep="last")

print(len(clean_claims_df))
business_key_duplicate_rows

## Your Turn 4: Check the lookup table key before joining

Read `policy_lookup.csv`. Show duplicate `policy_id` rows, then create `policy_clean_df` with one row per `policy_id`.

In [ ]:
policy_df = pd.read_csv(policy_path)
policy_duplicate_rows = policy_df[policy_df.duplicated(subset=["policy_id"], keep=False)]
policy_clean_df = policy_df.drop_duplicates(subset=["policy_id"])

policy_duplicate_rows

## Your Turn 5: Join and prove the row count

Left join clean claims to the clean lookup on `policy_id`. Print row counts before and after the join.

In [ ]:
joined_df = pd.merge(clean_claims_df, policy_clean_df, on="policy_id", how="left")

print(len(clean_claims_df))
print(len(joined_df))
joined_df

## Reflection

Which duplicate rule did you use for the claims table? Which duplicate rule did you use for the lookup table? Why are those rules different?

Your answer here.